In [1]:
import psycopg2

In [2]:
DATABASE_URL = "dbname=testdb user=testuser password=mypassword host=prototyping-pg-1"
TABLE_PREFIX = "v1"
PATCH_TABLE = f"{TABLE_PREFIX}_patch"
PRED_PATCH_TABLE = f"{TABLE_PREFIX}_pred_patch"

In [3]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

# Drop tables if they exist
cur.execute(f"DROP TABLE IF EXISTS {PATCH_TABLE} CASCADE;")
cur.execute(f"DROP TABLE IF EXISTS {PRED_PATCH_TABLE} CASCADE;")

conn.commit()
cur.close()
conn.close()

print(f"Tables {PATCH_TABLE} and {PRED_PATCH_TABLE} removed if they existed.")

Tables v1_patch and v1_pred_patch removed if they existed.


In [4]:


conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

create_table_query = f"""
CREATE TABLE {PATCH_TABLE}(
    id SERIAL NOT NULL,
    patch_uid integer NOT NULL,
    gt_label integer,
    event_ts timestamp with time zone NOT NULL DEFAULT now(),
    image_id integer,
    working_mag double precision,
    PRIMARY KEY(id)
);
"""

cur.execute(create_table_query)
conn.commit()
cur.close()
conn.close()

In [5]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

create_pred_patch_table_query = f"""
CREATE TABLE {PRED_PATCH_TABLE}(
    id SERIAL NOT NULL,
    patch_uid bigint NOT NULL,
    embed_coords point,
    grid_cell_i int,
    grid_cell_j int,
    event_ts timestamp with time zone NOT NULL DEFAULT now(),
    pred_label integer,
    patch_coords point,
    PRIMARY KEY(id)
);

CREATE INDEX idx_{PRED_PATCH_TABLE}_grid_cells ON {PRED_PATCH_TABLE} (grid_cell_i, grid_cell_j);
"""

cur.execute(create_pred_patch_table_query)
conn.commit()
cur.close()
conn.close()

In [6]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

# Finest level is 12 (4096x4096), coarsening up to level 9 (512x512)
# Divisor = 2^(finest_level - target_level)
levels = [
    ("l8", 16),
    ("l9",  8),   # level 9:  512x512   (divide l12 cells by 8)
    ("l10", 4),   # level 10: 1024x1024 (divide l12 cells by 4)
    ("l11", 2),   # level 11: 2048x2048 (divide l12 cells by 2)
    ("l12", 1),   # level 12: 4096x4096 (finest, no division)
]

statements = []
for level_name, divisor in levels:
    view_name = f"{TABLE_PREFIX}_patch_label_agg_{level_name}"
    index_name = f"idx_{TABLE_PREFIX}_patch_label_agg_{level_name}_grid"
    if divisor > 1:
        i_expr = f"(pp.grid_cell_i / {divisor})"
        j_expr = f"(pp.grid_cell_j / {divisor})"
    else:
        i_expr = "pp.grid_cell_i"
        j_expr = "pp.grid_cell_j"

    statements.append(f"""
CREATE MATERIALIZED VIEW {view_name} AS
SELECT
    pp.pred_label,
    p.gt_label,
    {i_expr} AS grid_cell_i,
    {j_expr} AS grid_cell_j,
    COUNT(*) AS patch_count
FROM {PRED_PATCH_TABLE} pp
LEFT JOIN {PATCH_TABLE} p ON pp.patch_uid = p.patch_uid
GROUP BY pp.pred_label, p.gt_label, {i_expr}, {j_expr};
""")
    statements.append(f"CREATE INDEX {index_name} ON {view_name} (grid_cell_i, grid_cell_j);")

for stmt in statements:
    cur.execute(stmt)

conn.commit()
cur.close()
conn.close()

print(f"Tables created: {PATCH_TABLE}, {PRED_PATCH_TABLE}")
print(f"Materialized views created: {TABLE_PREFIX}_patch_label_agg_l0 (coarsest) through {TABLE_PREFIX}_patch_label_agg_l4 (finest)")

Tables created: v1_patch, v1_pred_patch
Materialized views created: v1_patch_label_agg_l0 (coarsest) through v1_patch_label_agg_l4 (finest)


In [ ]:
import sys
import numpy as np
import psycopg2.extras

sys.path.insert(0, "/opt/PatchSorter/prototyping/tile_server_prototype")
from utils import HierarchicalGridIndexIJPair

N = 100_000
FINEST_LEVEL = 12  # 2^12 = 4096 cells per axis
GRID_SIZE = 2 ** FINEST_LEVEL  # 4096
NUM_LABELS = 5
rng = np.random.default_rng(42)

# --- Generate patch data ---
patch_uids   = np.arange(1, N + 1, dtype=int)
gt_labels    = rng.integers(0, NUM_LABELS, size=N).tolist()
image_ids    = rng.integers(1, 11, size=N).tolist()
working_mags = rng.choice([10.0, 20.0, 40.0], size=N).tolist()

# --- Generate embed coords: normal distribution clipped to [0, GRID_SIZE] ---
embed_x = np.clip(rng.normal(loc=GRID_SIZE / 2, scale=GRID_SIZE * 0.15, size=N), 0.0, GRID_SIZE)
embed_y = np.clip(rng.normal(loc=GRID_SIZE / 2, scale=GRID_SIZE * 0.15, size=N), 0.0, GRID_SIZE)

# --- Compute patch coords (random spatial locations, e.g. whole-slide pixels) ---
patch_x = rng.uniform(0, 10000, size=N)
patch_y = rng.uniform(0, 10000, size=N)

# --- Compute grid cells at finest level (12) using HierarchicalGridIndexIJPair ---
# cell_size=GRID_SIZE so that level 12 yields integer cell indices in [0, 4095]
grid = HierarchicalGridIndexIJPair(cell_size=GRID_SIZE)
pred_labels = rng.integers(0, NUM_LABELS, size=N).tolist()

grid_cells = [grid.point_to_cell(float(ex), float(ey), FINEST_LEVEL)
              for ex, ey in zip(embed_x, embed_y)]

# --- Bulk insert ---
conn = psycopg2.connect(DATABASE_URL)
cur  = conn.cursor()

patch_rows = [
    (int(patch_uids[i]), gt_labels[i], image_ids[i], working_mags[i])
    for i in range(N)
]
psycopg2.extras.execute_values(
    cur,
    f"INSERT INTO {PATCH_TABLE} (patch_uid, gt_label, image_id, working_mag) VALUES %s",
    patch_rows,
    page_size=1000,
)

pred_patch_rows = [
    (
        int(patch_uids[i]),
        f"({embed_x[i]},{embed_y[i]})",   # point literal
        int(grid_cells[i].i),
        int(grid_cells[i].j),
        pred_labels[i],
        f"({patch_x[i]},{patch_y[i]})",   # point literal
    )
    for i in range(N)
]
psycopg2.extras.execute_values(
    cur,
    f"""INSERT INTO {PRED_PATCH_TABLE}
        (patch_uid, embed_coords, grid_cell_i, grid_cell_j, pred_label, patch_coords)
        VALUES %s""",
    pred_patch_rows,
    page_size=1000,
)

conn.commit()
cur.close()
conn.close()

print(f"Inserted {N:,} rows into {PATCH_TABLE} and {PRED_PATCH_TABLE}")
print(f"Grid level: {FINEST_LEVEL} ({GRID_SIZE}x{GRID_SIZE}), cell_size: {grid.cell_size}")
print(f"Grid i range: [{min(c.i for c in grid_cells)}, {max(c.i for c in grid_cells)}]")
print(f"Grid j range: [{min(c.j for c in grid_cells)}, {max(c.j for c in grid_cells)}]")

Inserted 100,000 rows into v1_patch and v1_pred_patch
Grid level: 12 (4096x4096), cell_size: 4096
Grid i range: [0, 4096]
Grid j range: [0, 4096]


In [7]:
import sys
import numpy as np
import psycopg2.extras

sys.path.insert(0, "/opt/PatchSorter/prototyping/tile_server_prototype")
from utils import HierarchicalGridIndexIJPair

FINEST_LEVEL = 12
GRID_SIZE = 2 ** FINEST_LEVEL  # 4096
NUM_LABELS = 5
LARGE_N = 100_000
rng = np.random.default_rng(99)

grid = HierarchicalGridIndexIJPair(cell_size=GRID_SIZE)

def make_normal_points(n, mean_x, mean_y, scale=GRID_SIZE * 0.08):
    x = np.clip(rng.normal(loc=mean_x, scale=scale, size=n), 0.0, GRID_SIZE)
    y = np.clip(rng.normal(loc=mean_y, scale=scale, size=n), 0.0, GRID_SIZE)
    return x, y

def make_uniform_points(n):
    x = rng.uniform(0, GRID_SIZE, size=n)
    y = rng.uniform(0, GRID_SIZE, size=n)
    return x, y

def make_rows(embed_x, embed_y, gt_label, patch_uid_start):
    n = len(embed_x)
    pred_labels = rng.integers(0, NUM_LABELS, size=n)
    patch_x = rng.uniform(0, 10000, size=n)
    patch_y = rng.uniform(0, 10000, size=n)
    patch_uids = np.arange(patch_uid_start, patch_uid_start + n, dtype=int)
    grid_cells = [grid.point_to_cell(float(ex), float(ey), FINEST_LEVEL)
                  for ex, ey in zip(embed_x, embed_y)]

    patch_rows = [
        (int(patch_uids[i]), gt_label, None, None)
        for i in range(n)
    ]
    pred_patch_rows = [
        (
            int(patch_uids[i]),
            f"({embed_x[i]},{embed_y[i]})",
            int(grid_cells[i].i),
            int(grid_cells[i].j),
            int(pred_labels[i]),
            f"({patch_x[i]},{patch_y[i]})",
        )
        for i in range(n)
    ]
    return patch_rows, pred_patch_rows, patch_uid_start + n

all_patch_rows = []
all_pred_patch_rows = []
uid = 1

# --- gt_label=0: 4 normal distributions, 1M points each, random means ---
for _ in range(4):
    mean_x = rng.uniform(GRID_SIZE * 0.15, GRID_SIZE * 0.85)
    mean_y = rng.uniform(GRID_SIZE * 0.15, GRID_SIZE * 0.85)
    ex, ey = make_normal_points(LARGE_N, mean_x, mean_y)
    pr, ppr, uid = make_rows(ex, ey, gt_label=0, patch_uid_start=uid)
    all_patch_rows.extend(pr)
    all_pred_patch_rows.extend(ppr)

print(f"After gt_label=0: {len(all_patch_rows):,} rows, next uid={uid}")

# --- gt_label=1: 1 uniform distribution, 100 points ---
ex, ey = make_uniform_points(100)
pr, ppr, uid = make_rows(ex, ey, gt_label=1, patch_uid_start=uid)
all_patch_rows.extend(pr)
all_pred_patch_rows.extend(ppr)

print(f"After gt_label=1: {len(all_patch_rows):,} rows, next uid={uid}")

# --- gt_label=2: 100 uniform + 1M normal points ---
ex_u, ey_u = make_uniform_points(100)
mean_x = rng.uniform(GRID_SIZE * 0.15, GRID_SIZE * 0.85)
mean_y = rng.uniform(GRID_SIZE * 0.15, GRID_SIZE * 0.85)
ex_n, ey_n = make_normal_points(LARGE_N, mean_x, mean_y)
ex = np.concatenate([ex_u, ex_n])
ey = np.concatenate([ey_u, ey_n])
pr, ppr, uid = make_rows(ex, ey, gt_label=2, patch_uid_start=uid)
all_patch_rows.extend(pr)
all_pred_patch_rows.extend(ppr)

print(f"After gt_label=2: {len(all_patch_rows):,} rows, next uid={uid}")

# --- Bulk insert ---
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

psycopg2.extras.execute_values(
    cur,
    f"INSERT INTO {PATCH_TABLE} (patch_uid, gt_label, image_id, working_mag) VALUES %s",
    all_patch_rows,
    page_size=1000,
)

psycopg2.extras.execute_values(
    cur,
    f"""INSERT INTO {PRED_PATCH_TABLE}
        (patch_uid, embed_coords, grid_cell_i, grid_cell_j, pred_label, patch_coords)
        VALUES %s""",
    all_pred_patch_rows,
    page_size=1000,
)

conn.commit()
cur.close()
conn.close()

print(f"Inserted {len(all_patch_rows):,} total rows.")

After gt_label=0: 400,000 rows, next uid=400001
After gt_label=1: 400,100 rows, next uid=400101
After gt_label=2: 500,200 rows, next uid=500201
Inserted 500,200 total rows.


In [8]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

for level_name, _ in levels:
    view_name = f"{TABLE_PREFIX}_patch_label_agg_{level_name}"
    cur.execute(f"REFRESH MATERIALIZED VIEW {view_name};")

conn.commit()
cur.close()
conn.close()

print("Materialized views refreshed.")

Materialized views refreshed.


In [9]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

query = f"SELECT * FROM {view_name} LIMIT 10;"
cur.execute(query)

rows = cur.fetchall()
for row in rows:
    print(row)

cur.close()
conn.close()

(0, 0, 0, 1323, 1)
(0, 0, 0, 1394, 1)
(0, 0, 0, 1442, 1)
(0, 0, 0, 1461, 1)
(0, 0, 0, 1508, 1)
(0, 0, 0, 1593, 1)
(0, 0, 0, 1615, 1)
(0, 0, 0, 1620, 1)
(0, 0, 0, 1630, 1)
(0, 0, 0, 1649, 1)
